# Financial Equation Evaluation for Symbolic Regression

This notebook evaluates various financial equations using symbolic regression techniques to discover latent patterns in stock market data. We test different equation forms inspired by financial theory and market dynamics.

In [ ]:
import pandas as pd
import numpy as np
from scipy.optimize import minimize, curve_fit
from functools import partial
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

## Load Financial Datasets

In [ ]:
# Load the master financial dataset
df = pd.read_csv('csvs/financial_master_dataset.csv')
print(f"Loaded dataset with {len(df)} samples")
print(f"Unique tasks: {df['task'].unique()}")
print(f"Unique symbols: {df['symbol'].unique()}")

# Focus on SPY next-day return prediction for detailed analysis
spy_next_day = df[df['dataset_name'] == 'SPY_next_day_return'].copy()
print(f"\nSPY next-day return dataset: {spy_next_day.shape}")
spy_next_day.head()

## Define Financial Equation Hypotheses

Based on financial theory, we test several equation forms:
1. **Mean Reversion**: `target = c1 * (price - moving_average) + c2`
2. **Momentum**: `target = c1 * past_return + c2 * volatility + c3`
3. **Volume-Price**: `target = c1 * log(volume) + c2 * price_change + c3`
4. **Technical Analysis**: `target = c1 * RSI + c2 * volatility + c3`
5. **Complex Multi-factor**: More sophisticated combinations

In [ ]:
# Define equation functions
def mean_reversion_eq(x, c1, c2):
    """Mean reversion: return = c1 * (current_price - MA) + c2"""
    close, price_ma_20 = x
    return c1 * (close - price_ma_20) / price_ma_20 + c2

def momentum_eq(x, c1, c2, c3):
    """Momentum: return = c1 * volatility + c2 * RSI + c3"""
    volatility_5, rsi = x
    return c1 * volatility_5 + c2 * (rsi - 50) / 50 + c3

def volume_price_eq(x, c1, c2, c3):
    """Volume-Price: return = c1 * log(volume_ratio) + c2 * high_low_ratio + c3"""
    volume_ratio, high_low_ratio = x
    return c1 * np.log(volume_ratio + 1e-6) + c2 * (high_low_ratio - 1) + c3

def technical_analysis_eq(x, c1, c2, c3, c4):
    """Technical: return = c1 * RSI + c2 * volatility + c3 * volume_ratio + c4"""
    rsi, volatility_20, volume_ratio = x
    return c1 * (rsi - 50) / 50 + c2 * volatility_20 + c3 * np.log(volume_ratio + 1e-6) + c4

def complex_multifactor_eq(x, c1, c2, c3, c4, c5):
    """Complex: combines multiple factors with interactions"""
    close, price_ma_5, volatility_5, rsi, volume_ratio = x
    price_momentum = (close - price_ma_5) / price_ma_5
    rsi_normalized = (rsi - 50) / 50
    vol_factor = volatility_5
    volume_factor = np.log(volume_ratio + 1e-6)
    
    return (c1 * price_momentum + 
            c2 * rsi_normalized + 
            c3 * vol_factor + 
            c4 * volume_factor + 
            c5 * price_momentum * vol_factor)  # interaction term

# Equation metadata
equations = {
    'mean_reversion': {
        'function': mean_reversion_eq,
        'features': ['Close', 'Price_MA_20'],
        'params': 2,
        'description': 'c1 * (Close - Price_MA_20) / Price_MA_20 + c2'
    },
    'momentum': {
        'function': momentum_eq,
        'features': ['Volatility_5', 'RSI'],
        'params': 3,
        'description': 'c1 * Volatility_5 + c2 * (RSI - 50) / 50 + c3'
    },
    'volume_price': {
        'function': volume_price_eq,
        'features': ['Volume_Ratio', 'High_Low_Ratio'],
        'params': 3,
        'description': 'c1 * log(Volume_Ratio) + c2 * (High_Low_Ratio - 1) + c3'
    },
    'technical_analysis': {
        'function': technical_analysis_eq,
        'features': ['RSI', 'Volatility_20', 'Volume_Ratio'],
        'params': 4,
        'description': 'c1 * (RSI - 50) / 50 + c2 * Volatility_20 + c3 * log(Volume_Ratio) + c4'
    },
    'complex_multifactor': {
        'function': complex_multifactor_eq,
        'features': ['Close', 'Price_MA_5', 'Volatility_5', 'RSI', 'Volume_Ratio'],
        'params': 5,
        'description': 'Multi-factor with price momentum, RSI, volatility, volume, and interaction terms'
    }
}

print("Defined equation hypotheses:")
for name, eq_info in equations.items():
    print(f"{name}: {eq_info['description']}")

## Fit Equations to Financial Data

In [ ]:
def fit_equation(df, equation_name, equation_info, test_size=0.3):
    """Fit a financial equation to data and evaluate performance"""
    
    # Prepare data
    features = equation_info['features']
    X = df[features].values.T  # Transpose for equation functions
    y = df['target'].values
    
    # Remove NaN values
    valid_idx = ~(np.isnan(X).any(axis=0) | np.isnan(y))
    X = X[:, valid_idx]
    y = y[valid_idx]
    
    if len(y) < 50:  # Need minimum samples
        return None
    
    # Train-test split
    split_idx = int(len(y) * (1 - test_size))
    X_train, X_test = X[:, :split_idx], X[:, split_idx:]
    y_train, y_test = y[:split_idx], y[split_idx:]
    
    try:
        # Fit using curve_fit with bounds to prevent extreme parameters
        bounds = ([-10] * equation_info['params'], [10] * equation_info['params'])
        popt, pcov = curve_fit(equation_info['function'], X_train, y_train, 
                              bounds=bounds, maxfev=5000)
        
        # Evaluate on test set
        y_pred_test = equation_info['function'](X_test, *popt)
        y_pred_train = equation_info['function'](X_train, *popt)
        
        # Calculate metrics
        mse_train = mean_squared_error(y_train, y_pred_train)
        mse_test = mean_squared_error(y_test, y_pred_test)
        r2_train = r2_score(y_train, y_pred_train)
        r2_test = r2_score(y_test, y_pred_test)
        mae_test = mean_absolute_error(y_test, y_pred_test)
        
        return {
            'equation': equation_name,
            'parameters': popt,
            'mse_train': mse_train,
            'mse_test': mse_test,
            'r2_train': r2_train,
            'r2_test': r2_test,
            'mae_test': mae_test,
            'n_train': len(y_train),
            'n_test': len(y_test),
            'fitted_equation': equation_info['description'],
            'y_test': y_test,
            'y_pred_test': y_pred_test
        }
        
    except Exception as e:
        print(f"Failed to fit {equation_name}: {e}")
        return None

# Fit all equations to SPY next-day return data
results = []
fitted_results = {}

for eq_name, eq_info in equations.items():
    print(f"Fitting {eq_name}...")
    result = fit_equation(spy_next_day, eq_name, eq_info)
    if result:
        results.append(result)
        fitted_results[eq_name] = result
        print(f"  R² (test): {result['r2_test']:.4f}, MSE (test): {result['mse_test']:.6f}")

print(f"\nSuccessfully fitted {len(results)} equations")

## Equation Performance Comparison

In [ ]:
# Create results DataFrame
results_df = pd.DataFrame([
    {
        'Equation': r['equation'],
        'R² (Train)': r['r2_train'],
        'R² (Test)': r['r2_test'],
        'MSE (Train)': r['mse_train'],
        'MSE (Test)': r['mse_test'],
        'MAE (Test)': r['mae_test'],
        'Train Samples': r['n_train'],
        'Test Samples': r['n_test']
    }
    for r in results
])

# Sort by test R²
results_df = results_df.sort_values('R² (Test)', ascending=False)
print("Equation Performance Ranking:")
print(results_df.round(4))

In [ ]:
# Visualize performance
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# R² comparison
ax1 = axes[0, 0]
x_pos = np.arange(len(results_df))
ax1.bar(x_pos - 0.2, results_df['R² (Train)'], 0.4, label='Train', alpha=0.7)
ax1.bar(x_pos + 0.2, results_df['R² (Test)'], 0.4, label='Test', alpha=0.7)
ax1.set_xlabel('Equation')
ax1.set_ylabel('R² Score')
ax1.set_title('R² Score Comparison')
ax1.set_xticks(x_pos)
ax1.set_xticklabels(results_df['Equation'], rotation=45)
ax1.legend()
ax1.grid(True, alpha=0.3)

# MSE comparison
ax2 = axes[0, 1]
ax2.bar(x_pos - 0.2, results_df['MSE (Train)'], 0.4, label='Train', alpha=0.7)
ax2.bar(x_pos + 0.2, results_df['MSE (Test)'], 0.4, label='Test', alpha=0.7)
ax2.set_xlabel('Equation')
ax2.set_ylabel('MSE')
ax2.set_title('MSE Comparison')
ax2.set_xticks(x_pos)
ax2.set_xticklabels(results_df['Equation'], rotation=45)
ax2.legend()
ax2.grid(True, alpha=0.3)

# Prediction vs Actual for best model
best_model = results_df.iloc[0]['Equation']
best_result = fitted_results[best_model]

ax3 = axes[1, 0]
ax3.scatter(best_result['y_test'], best_result['y_pred_test'], alpha=0.6)
ax3.plot([best_result['y_test'].min(), best_result['y_test'].max()], 
         [best_result['y_test'].min(), best_result['y_test'].max()], 'r--', lw=2)
ax3.set_xlabel('Actual Returns')
ax3.set_ylabel('Predicted Returns')
ax3.set_title(f'Best Model: {best_model}')
ax3.grid(True, alpha=0.3)

# Residuals for best model
ax4 = axes[1, 1]
residuals = best_result['y_test'] - best_result['y_pred_test']
ax4.scatter(best_result['y_pred_test'], residuals, alpha=0.6)
ax4.axhline(y=0, color='r', linestyle='--')
ax4.set_xlabel('Predicted Returns')
ax4.set_ylabel('Residuals')
ax4.set_title(f'Residuals: {best_model}')
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Fitted Equation Details

In [ ]:
# Show fitted parameters for each equation
print("Fitted Equation Parameters:")
print("=" * 50)

for eq_name, result in fitted_results.items():
    print(f"\n{eq_name.upper()}:")
    print(f"Description: {result['fitted_equation']}")
    print(f"Parameters: {[f'{p:.4f}' for p in result['parameters']]}")
    print(f"Test R²: {result['r2_test']:.4f}")
    print(f"Test MSE: {result['mse_test']:.6f}")
    
    # Create fitted equation string
    eq_info = equations[eq_name]
    fitted_eq = eq_info['description']
    for i, param in enumerate(result['parameters']):
        fitted_eq = fitted_eq.replace(f'c{i+1}', f'{param:.4f}')
    print(f"Fitted equation: {fitted_eq}")

## Cross-Asset Analysis

In [ ]:
# Test equations on AAPL data
aapl_next_day = df[df['dataset_name'] == 'AAPL_next_day_return'].copy()
print(f"AAPL next-day return dataset: {aapl_next_day.shape}")

aapl_results = []
for eq_name, eq_info in equations.items():
    print(f"Fitting {eq_name} to AAPL...")
    result = fit_equation(aapl_next_day, eq_name, eq_info)
    if result:
        result['asset'] = 'AAPL'
        aapl_results.append(result)
        print(f"  R² (test): {result['r2_test']:.4f}, MSE (test): {result['mse_test']:.6f}")

# Compare performance across assets
cross_asset_comparison = []
for spy_result in results:
    spy_result['asset'] = 'SPY'
    cross_asset_comparison.append({
        'Equation': spy_result['equation'],
        'Asset': 'SPY',
        'R² (Test)': spy_result['r2_test'],
        'MSE (Test)': spy_result['mse_test']
    })

for aapl_result in aapl_results:
    cross_asset_comparison.append({
        'Equation': aapl_result['equation'],
        'Asset': 'AAPL',
        'R² (Test)': aapl_result['r2_test'],
        'MSE (Test)': aapl_result['mse_test']
    })

cross_asset_df = pd.DataFrame(cross_asset_comparison)
print("\nCross-Asset Performance Comparison:")
print(cross_asset_df.pivot(index='Equation', columns='Asset', values='R² (Test)').round(4))

## Save Results

In [ ]:
# Save detailed results
all_results = results + aapl_results
detailed_results = []

for result in all_results:
    detailed_results.append({
        'equation': result['equation'],
        'asset': result.get('asset', 'SPY'),
        'r2_train': result['r2_train'],
        'r2_test': result['r2_test'],
        'mse_train': result['mse_train'],
        'mse_test': result['mse_test'],
        'mae_test': result['mae_test'],
        'n_train': result['n_train'],
        'n_test': result['n_test'],
        'parameters': str(result['parameters'].tolist()),
        'description': result['fitted_equation']
    })

results_summary_df = pd.DataFrame(detailed_results)
results_summary_df.to_csv('csvs/equation_evaluation_results.csv', index=False)
print(f"Saved detailed results to csvs/equation_evaluation_results.csv")

# Save cross-asset comparison
cross_asset_df.to_csv('csvs/cross_asset_comparison.csv', index=False)
print(f"Saved cross-asset comparison to csvs/cross_asset_comparison.csv")

## Conclusions

This analysis has tested several financial equation hypotheses for predicting stock returns:

1. **Mean Reversion**: Tests if returns are predictable based on deviation from moving average
2. **Momentum**: Tests if volatility and RSI can predict future returns
3. **Volume-Price**: Tests volume and price range relationships
4. **Technical Analysis**: Combines multiple technical indicators
5. **Complex Multi-factor**: Tests interaction effects between factors

The results provide insights into which mathematical relationships might govern stock price movements and can guide the development of more sophisticated symbolic regression models for financial markets.

Key findings:
- Financial returns are inherently noisy and difficult to predict
- Different equation forms may work better for different assets
- Technical indicators show some predictive power but with low R² values
- Multi-factor models with interaction terms may capture more complex market dynamics